# 項目11: 個別銘柄による純化テーマファクター複製ポートフォリオ

この notebook は Phase B の追加項目として、個別銘柄ウェイトで純化テーマファクターリターンを複製するポートフォリオを構築する。

対象は、既に得られた Barra 既存ファクターエクスポージャ `X_t`、Barra ファクターリターン `f_t`、純化テーマファクターエクスポージャ `Z_t`、純化テーマリターン `g_t` を使う検証に限定する。テーマバスケット `A_t` への配分ではなく、個別銘柄へ直接ロングショート・ウェイトを置く。合成データは入れない。

主な制約は次の通り。

\[
1'w_k=0,\qquad X_t'w_k=0,\qquad z_{k,t}'w_k=1,\qquad Z_{-k,t}'w_k=0
\]

目的関数はリスク最小化とする。

\[
\min_{w_k}\;\frac{1}{2}w_k'\Sigma_t w_k
\]

`w_t` は構築日 `t` の情報だけで作り、複製精度の評価では `t` より後のリターンに適用する。

## 必要データ形式

| オブジェクト | 形式 | 必須列・index | 用途 |
|---|---:|---|---|
| `X_t` | DataFrame | index=`security_id`, columns=Barra factor | Barra 既存因子への中立制約 |
| `barra_factor_returns_f` | wide | index=営業日, columns=Barra factor | 複製リターンの寄与分解 |
| `Z_t` | DataFrame | index=`security_id`, columns=`theme_id` | 純化テーマへの単位エクスポージャ制約 |
| `theme_returns_g` | wide | index=営業日, columns=`theme_id` | 複製対象の純化テーマリターン |
| `stock_returns` | wide | index=営業日, columns=`security_id` | トータルリターンでの複製検証 |
| `barra_residuals_u` | wide | index=営業日, columns=`security_id` | Barra 残差リターンでの複製検証 |
| `Sigma_t` | DataFrame | index/columns=`security_id` | 銘柄リスク行列。利用可能な場合に使う |
| `specific_variance` | Series または wide | index=`security_id`、または index=営業日 × columns=`security_id` | `Sigma_t` が重い場合の対角リスク近似 |
| `purified_snapshots` | dict | `{date: {'X_t': ..., 'Z_t': ...}}` | 項目10の事前計算pklを使う場合 |

`Sigma_t` または `specific_variance` のどちらかは必須。`specific_variance` を使う場合、`Sigma_t = diag(specific_variance + ridge)` として扱う。

## ユーザ設定セル

下の `paths` または `existing_inputs` を実データに合わせて指定する。

- 項目10の事前計算pklを使う場合は、`purified_snapshots` と `theme_return_history` を指定する。
- 単日スナップショットだけで確認する場合は、`X_t`, `Z_t`, `specific_variance` または `Sigma_t` を指定する。
- 履歴評価を行う場合は `RUN_HISTORY_EVALUATION = True` にし、`stock_returns`, `barra_residuals_u`, `theme_returns_g` も指定する。

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any, Mapping

import numpy as np
import pandas as pd
from IPython.display import display


@dataclass(frozen=True)
class FactorMimickingConfig:
    asof_date: str | pd.Timestamp = "YYYY-MM-DD"
    neutralize_other_themes: bool = True
    pinv_rcond: float = 1e-10
    ridge: float = 1e-8
    constraint_tol: float = 1e-6
    min_abs_target_exposure: float = 1e-12
    continue_on_error: bool = True
    evaluation_start: str | pd.Timestamp | None = None
    evaluation_end: str | pd.Timestamp | None = None
    annualization_days: int = 252


PROJECT_ROOT = Path.cwd()
cfg = FactorMimickingConfig(asof_date="YYYY-MM-DD")

paths = {
    # 項目10の事前計算pklを使う場合
    "purified_snapshots": "data/phase_b/precomputed/item10/purified_snapshots.pkl",
    "theme_return_history": "data/phase_b/precomputed/item10/theme_return_history.pkl",

    # 単日スナップショットを直接使う場合
    "X_t": "data/phase_b/X_t.pkl",
    "Z_t": "data/phase_b/Z_t.pkl",
    "Sigma_t": "",  # full covariance を使う場合のみ指定
    "specific_variance": "data/phase_b/specific_variance.pkl",

    # 複製精度の履歴評価に使う入力
    "stock_returns": "data/phase_b/stock_returns.pkl",
    "barra_residuals_u": "data/phase_b/barra_residuals_u.pkl",
    "theme_returns_g": "data/phase_b/theme_returns_g.pkl",
    "barra_factor_returns_f": "data/phase_b/barra_factor_returns_f.pkl",
}

# 既にNotebook上にDataFrameやdictを作っている場合はここへ渡す。
# 例: existing_inputs = {"X_t": X_t, "Z_t": Z_t, "specific_variance": specific_variance}
existing_inputs: dict[str, Any] = {}

RUN_HISTORY_EVALUATION = False


## 入力ロード・正規化ユーティリティ

In [ ]:
def _as_date(value: str | pd.Timestamp) -> pd.Timestamp:
    return pd.Timestamp(value).normalize()


def _normalise_date_index(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.index = pd.to_datetime(out.index).normalize()
    return out.sort_index()


def _resolve_path(path: str | Path | None) -> Path | None:
    if path is None or str(path).strip() == "":
        return None
    p = Path(path)
    return p if p.is_absolute() else PROJECT_ROOT / p


def _read_pickle(path: str | Path) -> Any:
    p = _resolve_path(path)
    if p is None:
        raise ValueError("path is empty.")
    if not p.exists():
        raise FileNotFoundError(f"Input file not found: {p}")
    return pd.read_pickle(p)


def _load_input(name: str, required: bool = True) -> Any:
    if name in existing_inputs:
        return existing_inputs[name]
    path = paths.get(name)
    p = _resolve_path(path)
    if p is not None and p.exists():
        return _read_pickle(p)
    if required:
        raise FileNotFoundError(
            f"{name} が見つかりません。paths['{name}'] または existing_inputs['{name}'] を指定してください。"
        )
    return None


def _normalise_snapshot_keys(snapshots: Mapping[Any, Any]) -> dict[pd.Timestamp, Any]:
    out: dict[pd.Timestamp, Any] = {}
    for key, value in snapshots.items():
        out[_as_date(key)] = value
    return out


def _get_snapshot_for_date(snapshots: Mapping[Any, Any], date: str | pd.Timestamp) -> dict[str, Any]:
    d = _as_date(date)
    normalised = _normalise_snapshot_keys(snapshots)
    if d not in normalised:
        raise KeyError(f"purified_snapshots に {d.date()} のsnapshotがありません。forward-fillは行いません。")
    snap = normalised[d]
    if not isinstance(snap, Mapping):
        raise TypeError("snapshot must be a mapping with keys such as 'X_t' and 'Z_t'.")
    if "X_t" not in snap or "Z_t" not in snap:
        raise KeyError("snapshot must contain 'X_t' and 'Z_t'.")
    return dict(snap)


def _select_date_object(obj: Any, date: str | pd.Timestamp, name: str) -> Any:
    d = _as_date(date)
    if obj is None:
        return None
    if isinstance(obj, Mapping):
        normalised = _normalise_snapshot_keys(obj)
        if d not in normalised:
            raise KeyError(f"{name} に {d.date()} のデータがありません。")
        return normalised[d]
    if isinstance(obj, pd.DataFrame):
        if isinstance(obj.index, pd.DatetimeIndex):
            df = _normalise_date_index(obj)
            if d not in df.index:
                raise KeyError(f"{name} に {d.date()} の行がありません。")
            return df.loc[d]
        return obj
    if isinstance(obj, pd.Series):
        if isinstance(obj.index, pd.DatetimeIndex):
            s = obj.copy()
            s.index = pd.to_datetime(s.index).normalize()
            if d not in s.index:
                raise KeyError(f"{name} に {d.date()} の値がありません。")
            return s.loc[d]
        return obj
    return obj


def _load_theme_history() -> dict[str, Any]:
    history = _load_input("theme_return_history", required=False)
    if history is None:
        history = {}
    if not isinstance(history, Mapping):
        raise TypeError("theme_return_history must be a dict-like object.")

    out = dict(history)
    for key in ["theme_returns_g", "barra_residuals_u", "barra_factor_returns_f"]:
        if key not in out:
            loaded = _load_input(key, required=False)
            if loaded is not None:
                out[key] = loaded
    return out


def load_asof_snapshot(cfg: FactorMimickingConfig) -> dict[str, pd.DataFrame]:
    snapshots = _load_input("purified_snapshots", required=False)
    if snapshots is not None:
        snap = _get_snapshot_for_date(snapshots, cfg.asof_date)
        X_t = snap["X_t"].copy()
        Z_t = snap["Z_t"].copy()
    else:
        X_t = _load_input("X_t", required=True).copy()
        Z_t = _load_input("Z_t", required=True).copy()

    X_t.index = X_t.index.astype(str)
    Z_t.index = Z_t.index.astype(str)
    X_t.columns = X_t.columns.astype(str)
    Z_t.columns = Z_t.columns.astype(str)
    X_t = X_t.sort_index().astype(float)
    Z_t = Z_t.sort_index().astype(float)
    return {"X_t": X_t, "Z_t": Z_t}


## リスク行列の構築

`Sigma_t` が利用できる場合は full covariance を使う。利用できない場合は `specific_variance` から対角リスク行列を作る。どちらもない場合は停止する。

In [ ]:
def _safe_condition_number(matrix: np.ndarray) -> float:
    if matrix.size == 0:
        return np.nan
    try:
        return float(np.linalg.cond(matrix))
    except np.linalg.LinAlgError:
        return np.inf


def _finite_square_mask(df: pd.DataFrame) -> pd.Index:
    arr = df.to_numpy(dtype=float)
    finite_rows = np.isfinite(arr).all(axis=1)
    finite_cols = np.isfinite(arr).all(axis=0)
    diag_ok = np.isfinite(np.diag(arr)) & (np.diag(arr) > 0)
    mask = finite_rows & finite_cols & diag_ok
    return df.index[mask]


def _risk_matrix_from_covariance(Sigma_t: pd.DataFrame, securities: pd.Index, cfg: FactorMimickingConfig) -> pd.DataFrame:
    Sigma = Sigma_t.copy()
    Sigma.index = Sigma.index.astype(str)
    Sigma.columns = Sigma.columns.astype(str)
    common = securities.intersection(Sigma.index).intersection(Sigma.columns)
    if common.empty:
        raise ValueError("Sigma_t と X_t/Z_t の共通銘柄が空です。")
    Sigma = Sigma.loc[common, common].astype(float)
    valid = _finite_square_mask(Sigma)
    Sigma = Sigma.loc[valid, valid]
    if Sigma.empty:
        raise ValueError("Sigma_t の有効な銘柄集合が空です。")
    Sigma = (Sigma + Sigma.T) / 2.0
    Sigma = Sigma + np.eye(len(Sigma)) * cfg.ridge
    return Sigma


def _risk_matrix_from_specific_variance(
    specific_variance: pd.Series | pd.DataFrame,
    securities: pd.Index,
    cfg: FactorMimickingConfig,
) -> pd.DataFrame:
    if isinstance(specific_variance, pd.DataFrame):
        if specific_variance.shape[1] == 1:
            s = specific_variance.iloc[:, 0]
        else:
            raise ValueError("specific_variance DataFrameは、日付選択後はSeriesまたは1列DataFrameにしてください。")
    else:
        s = pd.Series(specific_variance)
    s.index = s.index.astype(str)
    s = s.reindex(securities).astype(float)
    valid = s.notna() & np.isfinite(s.to_numpy(dtype=float)) & (s > 0)
    s = s.loc[valid] + cfg.ridge
    if s.empty:
        raise ValueError("specific_variance の有効な銘柄集合が空です。")
    return pd.DataFrame(np.diag(s.to_numpy(dtype=float)), index=s.index, columns=s.index)


def load_risk_matrix_for_date(securities: pd.Index, date: str | pd.Timestamp, cfg: FactorMimickingConfig) -> pd.DataFrame:
    sigma_input = _load_input("Sigma_t", required=False)
    specific_input = _load_input("specific_variance", required=False)

    if sigma_input is not None:
        Sigma_t = _select_date_object(sigma_input, date, "Sigma_t")
        if not isinstance(Sigma_t, pd.DataFrame):
            raise TypeError("Sigma_t must be a square DataFrame after date selection.")
        return _risk_matrix_from_covariance(Sigma_t, securities, cfg)

    if specific_input is not None:
        specific_t = _select_date_object(specific_input, date, "specific_variance")
        return _risk_matrix_from_specific_variance(specific_t, securities, cfg)

    raise FileNotFoundError("Sigma_t または specific_variance を指定してください。")


## 複製ポートフォリオの解析解

各テーマ列について、制約行列 `C_k = [1, X_t, Z_{-k,t}, z_{k,t}]` と目標ベクトル `d_k = [0, 0, ..., 0, 1]` を作り、以下を一般化逆行列で解く。

\[
w_k=\Sigma_t^{-1}C_k(C_k'\Sigma_t^{-1}C_k)^+d_k
\]

rank不足や多重共線性は停止理由にせず、制約誤差・rank・条件数を診断に残す。

In [ ]:
def _align_model_inputs(
    X_t: pd.DataFrame,
    Z_t: pd.DataFrame,
    Sigma_t: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    common = X_t.index.intersection(Z_t.index).intersection(Sigma_t.index).intersection(Sigma_t.columns)
    if common.empty:
        raise ValueError("X_t, Z_t, Sigma_t の共通銘柄が空です。")

    X = X_t.loc[common].astype(float)
    Z = Z_t.loc[common].astype(float)
    Sigma = Sigma_t.loc[common, common].astype(float)

    finite_x = np.isfinite(X.to_numpy(dtype=float)).all(axis=1)
    finite_z = np.isfinite(Z.to_numpy(dtype=float)).all(axis=1)
    finite_sigma = np.isfinite(Sigma.to_numpy(dtype=float)).all(axis=1)
    valid = pd.Index(common[finite_x & finite_z & finite_sigma])
    if valid.empty:
        raise ValueError("欠損除外後の有効銘柄が空です。")

    X = X.loc[valid]
    Z = Z.loc[valid]
    Sigma = Sigma.loc[valid, valid]
    return X, Z, Sigma


def solve_one_mimicking_portfolio(
    X_t: pd.DataFrame,
    Z_t: pd.DataFrame,
    Sigma_t: pd.DataFrame,
    theme_id: str,
    cfg: FactorMimickingConfig,
) -> dict[str, Any]:
    if theme_id not in Z_t.columns:
        raise KeyError(f"theme_id={theme_id} is not in Z_t columns.")

    X, Z, Sigma = _align_model_inputs(X_t, Z_t, Sigma_t)
    if abs(float(np.linalg.norm(Z[theme_id].to_numpy(dtype=float)))) <= cfg.min_abs_target_exposure:
        raise ValueError(f"target theme exposure is effectively zero: {theme_id}")

    constraint_parts = [pd.Series(1.0, index=X.index, name="net").to_frame()]
    constraint_parts.append(X.add_prefix("barra:"))

    other_themes = [c for c in Z.columns if c != theme_id]
    if cfg.neutralize_other_themes and other_themes:
        constraint_parts.append(Z.loc[:, other_themes].add_prefix("other_theme:"))

    target_name = f"target_theme:{theme_id}"
    constraint_parts.append(Z[[theme_id]].rename(columns={theme_id: target_name}))
    C = pd.concat(constraint_parts, axis=1).astype(float)

    d = pd.Series(0.0, index=C.columns)
    d.loc[target_name] = 1.0

    Sigma_np = Sigma.to_numpy(dtype=float)
    C_np = C.to_numpy(dtype=float)
    d_np = d.to_numpy(dtype=float)

    Sigma_inv = np.linalg.pinv(Sigma_np, rcond=cfg.pinv_rcond)
    Sigma_inv_C = Sigma_inv @ C_np
    middle = C_np.T @ Sigma_inv_C
    multipliers = np.linalg.pinv(middle, rcond=cfg.pinv_rcond) @ d_np
    w_np = Sigma_inv_C @ multipliers

    weights = pd.Series(w_np, index=X.index, name=theme_id)
    constraint_values = pd.Series(C_np.T @ w_np, index=C.columns, name="value")
    constraint_errors = constraint_values - d

    barra_exposure = pd.Series(X.to_numpy(dtype=float).T @ w_np, index=X.columns, name=theme_id)
    theme_exposure = pd.Series(Z.to_numpy(dtype=float).T @ w_np, index=Z.columns, name=theme_id)

    variance = float(w_np @ Sigma_np @ w_np)
    gross = float(weights.abs().sum())
    long_exposure = float(weights.clip(lower=0.0).sum())
    short_exposure = float(-weights.clip(upper=0.0).sum())
    abs_weight = weights.abs()
    concentration_hhi = float(((abs_weight / gross) ** 2).sum()) if gross > 0 else np.nan

    other_abs = theme_exposure.drop(theme_id, errors="ignore").abs()
    diagnostics = pd.Series(
        {
            "theme_id": theme_id,
            "n_securities": len(weights),
            "n_barra_factors": X.shape[1],
            "n_theme_factors": Z.shape[1],
            "n_constraints": C.shape[1],
            "rank_C": int(np.linalg.matrix_rank(C_np)),
            "rank_middle": int(np.linalg.matrix_rank(middle)),
            "condition_C": _safe_condition_number(C_np),
            "condition_middle": _safe_condition_number(middle),
            "net_exposure": float(weights.sum()),
            "gross_exposure": gross,
            "long_exposure": long_exposure,
            "short_exposure": short_exposure,
            "portfolio_variance": variance,
            "portfolio_risk": float(np.sqrt(max(variance, 0.0))),
            "max_abs_barra_exposure": float(barra_exposure.abs().max()) if not barra_exposure.empty else 0.0,
            "target_theme_exposure": float(theme_exposure.loc[theme_id]),
            "target_exposure_error_abs": float(abs(theme_exposure.loc[theme_id] - 1.0)),
            "max_abs_other_theme_exposure": float(other_abs.max()) if not other_abs.empty else 0.0,
            "max_abs_constraint_error": float(constraint_errors.abs().max()),
            "concentration_hhi_abs_weight": concentration_hhi,
        }
    )

    constraint_table = pd.DataFrame({"target": d, "value": constraint_values, "error": constraint_errors})
    return {
        "weights": weights,
        "diagnostics": diagnostics,
        "constraints": constraint_table,
        "barra_exposure": barra_exposure,
        "theme_exposure": theme_exposure,
    }


def solve_theme_mimicking_portfolios(
    X_t: pd.DataFrame,
    Z_t: pd.DataFrame,
    Sigma_t: pd.DataFrame,
    cfg: FactorMimickingConfig,
) -> dict[str, Any]:
    weight_columns: dict[str, pd.Series] = {}
    diagnostics_rows: list[pd.Series] = []
    constraint_tables: dict[str, pd.DataFrame] = {}
    barra_exposures: dict[str, pd.Series] = {}
    theme_exposures: dict[str, pd.Series] = {}
    errors: dict[str, str] = {}

    for theme_id in Z_t.columns.astype(str):
        try:
            result = solve_one_mimicking_portfolio(X_t, Z_t, Sigma_t, theme_id, cfg)
            weight_columns[theme_id] = result["weights"]
            diagnostics_rows.append(result["diagnostics"])
            constraint_tables[theme_id] = result["constraints"]
            barra_exposures[theme_id] = result["barra_exposure"]
            theme_exposures[theme_id] = result["theme_exposure"]
        except Exception as exc:
            errors[theme_id] = f"{type(exc).__name__}: {exc}"
            if not cfg.continue_on_error:
                raise

    weights = pd.DataFrame(weight_columns).fillna(0.0).sort_index()
    diagnostics = pd.DataFrame(diagnostics_rows).set_index("theme_id") if diagnostics_rows else pd.DataFrame()
    barra_exposure = pd.DataFrame(barra_exposures).T if barra_exposures else pd.DataFrame()
    theme_exposure = pd.DataFrame(theme_exposures).T if theme_exposures else pd.DataFrame()
    error_series = pd.Series(errors, name="error", dtype="object")

    return {
        "mimicking_weights": weights,
        "mimicking_diagnostics": diagnostics,
        "constraint_tables": constraint_tables,
        "barra_exposure": barra_exposure,
        "theme_exposure": theme_exposure,
        "errors": error_series,
    }


## 単日スナップショットでの実行

`asof_date` の `X_t`, `Z_t` とリスク入力から、テーマ別の個別銘柄複製ウェイトを作る。

In [ ]:
snapshot = load_asof_snapshot(cfg)
X_t = snapshot["X_t"]
Z_t = snapshot["Z_t"]
Sigma_t = load_risk_matrix_for_date(X_t.index.intersection(Z_t.index), cfg.asof_date, cfg)

mimicking_result = solve_theme_mimicking_portfolios(X_t, Z_t, Sigma_t, cfg)

mimicking_weights_t = mimicking_result["mimicking_weights"]
mimicking_diagnostics_t = mimicking_result["mimicking_diagnostics"]
barra_exposure_t = mimicking_result["barra_exposure"]
theme_exposure_t = mimicking_result["theme_exposure"]
mimicking_errors_t = mimicking_result["errors"]

print(f"X_t shape: {X_t.shape}")
print(f"Z_t shape: {Z_t.shape}")
print(f"Sigma_t shape: {Sigma_t.shape}")
print(f"mimicking_weights_t shape: {mimicking_weights_t.shape}")
print(f"solved themes: {mimicking_weights_t.shape[1]}")
if not mimicking_errors_t.empty:
    print("Errors:")
    display(mimicking_errors_t)


## 出力確認

In [ ]:
display(mimicking_diagnostics_t)
display(barra_exposure_t.head())
display(theme_exposure_t.head())
display(mimicking_weights_t.head())


## 寄与分解

Barra ファクターリターン `f_t` と純化テーマリターン `g_t` がある場合、構築したポートフォリオのファクター寄与を確認する。Barra中立が効いていれば、Barra寄与はゼロ近傍になる。

In [ ]:
def _select_history_row(frame: pd.DataFrame | pd.Series, date: str | pd.Timestamp, name: str) -> pd.Series:
    selected = _select_date_object(frame, date, name)
    if isinstance(selected, pd.DataFrame):
        if selected.shape[0] == 1:
            return selected.iloc[0]
        if selected.shape[1] == 1:
            return selected.iloc[:, 0]
        raise ValueError(f"{name} の日付選択後はSeriesまたは1行/1列DataFrameである必要があります。")
    return pd.Series(selected)


def decompose_factor_contributions(
    weights_t: pd.DataFrame,
    X_t: pd.DataFrame,
    Z_t: pd.DataFrame,
    barra_factor_returns_f_t: pd.Series,
    theme_returns_g_t: pd.Series,
) -> dict[str, pd.Series | pd.DataFrame]:
    securities = weights_t.index.intersection(X_t.index).intersection(Z_t.index)
    themes = weights_t.columns.intersection(Z_t.columns).intersection(theme_returns_g_t.index)
    factors = X_t.columns.intersection(barra_factor_returns_f_t.index)
    if securities.empty or themes.empty:
        raise ValueError("寄与分解に必要な共通銘柄またはテーマが空です。")

    W = weights_t.loc[securities, themes].astype(float)
    X = X_t.loc[securities, factors].astype(float)
    Z = Z_t.loc[securities, themes].astype(float)
    f = barra_factor_returns_f_t.reindex(factors).astype(float)
    g = theme_returns_g_t.reindex(themes).astype(float)

    barra_exp = X.T @ W
    theme_exp = Z.T @ W
    barra_contribution = barra_exp.T @ f
    theme_contribution = theme_exp.T @ g
    target_theme_contribution = pd.Series(
        {theme: float(theme_exp.loc[theme, theme] * g.loc[theme]) for theme in themes},
        name="target_theme_contribution",
    )

    return {
        "barra_exposure": barra_exp,
        "theme_exposure": theme_exp,
        "barra_contribution": barra_contribution.rename("barra_contribution"),
        "theme_contribution": theme_contribution.rename("theme_contribution"),
        "target_theme_contribution": target_theme_contribution,
    }


history = _load_theme_history()
if "barra_factor_returns_f" in history and "theme_returns_g" in history:
    f_asof = _select_history_row(history["barra_factor_returns_f"], cfg.asof_date, "barra_factor_returns_f")
    g_asof = _select_history_row(history["theme_returns_g"], cfg.asof_date, "theme_returns_g")
    contribution_result = decompose_factor_contributions(mimicking_weights_t, X_t, Z_t, f_asof, g_asof)
    display(contribution_result["barra_contribution"].to_frame())
    display(contribution_result["theme_contribution"].to_frame())
    display(contribution_result["target_theme_contribution"].to_frame())
else:
    contribution_result = None
    print("barra_factor_returns_f または theme_returns_g が未指定のため、寄与分解はスキップしました。")


## 履歴ウェイト構築と複製リターン評価

`RUN_HISTORY_EVALUATION = True` の場合、`purified_snapshots` の各日付でウェイトを構築し、構築日より後のリターンに適用して `g_t` との複製精度を評価する。

In [ ]:
def build_mimicking_weight_history(
    purified_snapshots: Mapping[Any, Any],
    cfg: FactorMimickingConfig,
    dates: list[pd.Timestamp] | None = None,
) -> dict[str, Any]:
    snapshots = _normalise_snapshot_keys(purified_snapshots)
    run_dates = sorted(dates if dates is not None else snapshots.keys())

    weight_frames: list[pd.DataFrame] = []
    diagnostic_frames: list[pd.DataFrame] = []
    errors: dict[pd.Timestamp, str] = {}

    for d in run_dates:
        try:
            snap = _get_snapshot_for_date(snapshots, d)
            X_d = snap["X_t"].copy()
            Z_d = snap["Z_t"].copy()
            X_d.index = X_d.index.astype(str)
            Z_d.index = Z_d.index.astype(str)
            X_d.columns = X_d.columns.astype(str)
            Z_d.columns = Z_d.columns.astype(str)
            X_d = X_d.astype(float)
            Z_d = Z_d.astype(float)
            Sigma_d = load_risk_matrix_for_date(X_d.index.intersection(Z_d.index), d, cfg)
            result = solve_theme_mimicking_portfolios(X_d, Z_d, Sigma_d, cfg)

            W = result["mimicking_weights"]
            W.index = pd.MultiIndex.from_product([[pd.Timestamp(d)], W.index], names=["date", "security_id"])
            weight_frames.append(W)

            diag = result["mimicking_diagnostics"].copy()
            if not diag.empty:
                diag.insert(0, "date", pd.Timestamp(d))
                diagnostic_frames.append(diag.reset_index().set_index(["date", "theme_id"]))
        except Exception as exc:
            errors[pd.Timestamp(d)] = f"{type(exc).__name__}: {exc}"
            if not cfg.continue_on_error:
                raise

    weights = pd.concat(weight_frames).sort_index() if weight_frames else pd.DataFrame()
    diagnostics = pd.concat(diagnostic_frames).sort_index() if diagnostic_frames else pd.DataFrame()
    return {"mimicking_weights": weights, "mimicking_diagnostics": diagnostics, "errors": pd.Series(errors, dtype="object")}


def _date_window(index: pd.DatetimeIndex, cfg: FactorMimickingConfig) -> pd.DatetimeIndex:
    dates = pd.DatetimeIndex(index).normalize().sort_values().unique()
    if cfg.evaluation_start is not None:
        dates = dates[dates >= _as_date(cfg.evaluation_start)]
    if cfg.evaluation_end is not None:
        dates = dates[dates <= _as_date(cfg.evaluation_end)]
    return dates


def _replication_metrics(replicated: pd.DataFrame, target: pd.DataFrame, annualization_days: int) -> pd.DataFrame:
    rows: list[dict[str, float | str]] = []
    for theme in replicated.columns.intersection(target.columns):
        aligned = pd.concat([replicated[theme], target[theme]], axis=1, keys=["replicated", "target"]).dropna()
        if aligned.empty:
            rows.append({"theme_id": theme, "observations": 0})
            continue
        diff = aligned["replicated"] - aligned["target"]
        target_var = float(aligned["target"].var(ddof=1)) if len(aligned) > 1 else np.nan
        beta = float(aligned.cov().loc["replicated", "target"] / target_var) if target_var and np.isfinite(target_var) and target_var > 0 else np.nan
        corr = float(aligned["replicated"].corr(aligned["target"])) if len(aligned) > 1 else np.nan
        rows.append(
            {
                "theme_id": theme,
                "observations": len(aligned),
                "correlation": corr,
                "beta_to_target": beta,
                "r2_corr_squared": corr ** 2 if np.isfinite(corr) else np.nan,
                "mean_error_daily": float(diff.mean()),
                "tracking_error_daily": float(diff.std(ddof=1)) if len(diff) > 1 else np.nan,
                "tracking_error_annualized": float(diff.std(ddof=1) * np.sqrt(annualization_days)) if len(diff) > 1 else np.nan,
            }
        )
    return pd.DataFrame(rows).set_index("theme_id") if rows else pd.DataFrame()


def evaluate_replicated_returns(
    mimicking_weights: pd.DataFrame,
    stock_returns: pd.DataFrame,
    barra_residuals_u: pd.DataFrame,
    theme_returns_g: pd.DataFrame,
    cfg: FactorMimickingConfig,
) -> dict[str, pd.DataFrame]:
    if not isinstance(mimicking_weights.index, pd.MultiIndex):
        raise TypeError("mimicking_weights must have MultiIndex index=(date, security_id).")

    stock = _normalise_date_index(stock_returns)
    residual = _normalise_date_index(barra_residuals_u)
    target = _normalise_date_index(theme_returns_g)
    stock.columns = stock.columns.astype(str)
    residual.columns = residual.columns.astype(str)

    weight_dates = pd.DatetimeIndex(mimicking_weights.index.get_level_values("date").unique()).normalize().sort_values()
    return_dates = _date_window(stock.index.intersection(residual.index).intersection(target.index), cfg)

    total_rows: list[pd.Series] = []
    residual_rows: list[pd.Series] = []
    coverage_rows: list[pd.Series] = []

    for d in return_dates:
        candidates = weight_dates[weight_dates < d]
        if len(candidates) == 0:
            continue
        weight_date = candidates[-1]
        W = mimicking_weights.xs(weight_date, level="date")
        themes = W.columns.intersection(target.columns)
        securities = W.index.intersection(stock.columns).intersection(residual.columns)
        if securities.empty or themes.empty:
            continue

        W = W.loc[securities, themes].astype(float)
        r = stock.loc[d, securities].astype(float)
        u = residual.loc[d, securities].astype(float)

        total_valid = r.notna() & np.isfinite(r.to_numpy(dtype=float))
        residual_valid = u.notna() & np.isfinite(u.to_numpy(dtype=float))

        total = (W.loc[total_valid].mul(r.loc[total_valid], axis=0)).sum(axis=0)
        resid = (W.loc[residual_valid].mul(u.loc[residual_valid], axis=0)).sum(axis=0)

        total.name = d
        resid.name = d
        total_rows.append(total)
        residual_rows.append(resid)

        gross = W.abs().sum(axis=0).replace(0.0, np.nan)
        total_cov = W.loc[total_valid].abs().sum(axis=0) / gross
        residual_cov = W.loc[residual_valid].abs().sum(axis=0) / gross
        coverage = pd.concat(
            [total_cov.rename("total_return_abs_weight_coverage"), residual_cov.rename("residual_abs_weight_coverage")],
            axis=1,
        ).stack()
        coverage.name = d
        coverage_rows.append(coverage)

    replicated_total = pd.DataFrame(total_rows).sort_index() if total_rows else pd.DataFrame()
    replicated_residual = pd.DataFrame(residual_rows).sort_index() if residual_rows else pd.DataFrame()
    target_aligned = target.reindex(replicated_residual.index).reindex(columns=replicated_residual.columns)
    coverage = pd.DataFrame(coverage_rows).sort_index() if coverage_rows else pd.DataFrame()

    residual_metrics = _replication_metrics(replicated_residual, target_aligned, cfg.annualization_days)
    total_metrics = _replication_metrics(replicated_total, target_aligned, cfg.annualization_days)

    return {
        "replicated_returns_total": replicated_total,
        "replicated_returns_residual": replicated_residual,
        "target_theme_returns_g": target_aligned,
        "replication_metrics_residual": residual_metrics,
        "replication_metrics_total": total_metrics,
        "daily_weight_coverage": coverage,
    }


if RUN_HISTORY_EVALUATION:
    purified_snapshots = _load_input("purified_snapshots", required=True)
    history_result = build_mimicking_weight_history(purified_snapshots, cfg)
    mimicking_weights = history_result["mimicking_weights"]
    mimicking_diagnostics = history_result["mimicking_diagnostics"]

    history = _load_theme_history()
    stock_returns = _load_input("stock_returns", required=True)
    barra_residuals_u = history.get("barra_residuals_u", _load_input("barra_residuals_u", required=True))
    theme_returns_g = history.get("theme_returns_g", _load_input("theme_returns_g", required=True))

    replication_result = evaluate_replicated_returns(
        mimicking_weights,
        stock_returns,
        barra_residuals_u,
        theme_returns_g,
        cfg,
    )

    replicated_returns_total = replication_result["replicated_returns_total"]
    replicated_returns_residual = replication_result["replicated_returns_residual"]
    target_theme_returns_g = replication_result["target_theme_returns_g"]

    display(replication_result["replication_metrics_residual"])
    display(replication_result["replication_metrics_total"])
else:
    print("RUN_HISTORY_EVALUATION=False のため、履歴評価はスキップしました。")


## 受入チェック

単日スナップショットについて、対象テーマ単位エクスポージャ、ネットゼロ、Barra中立、他テーマ中立を確認する。

In [ ]:
assert not mimicking_weights_t.empty, "mimicking_weights_t is empty."
assert not mimicking_diagnostics_t.empty, "mimicking_diagnostics_t is empty."

assert mimicking_diagnostics_t["target_exposure_error_abs"].max() < cfg.constraint_tol, "z_k' w_k must be close to 1."
assert mimicking_diagnostics_t["net_exposure"].abs().max() < cfg.constraint_tol, "sum(w_k) must be close to 0."
assert mimicking_diagnostics_t["max_abs_barra_exposure"].max() < cfg.constraint_tol, "X_t' w_k must be close to 0."

if cfg.neutralize_other_themes and Z_t.shape[1] > 1:
    assert mimicking_diagnostics_t["max_abs_other_theme_exposure"].max() < cfg.constraint_tol, "Z_{-k}' w_k must be close to 0."

print("Acceptance checks passed for the asof snapshot.")


## 次の確認ポイント

- `mimicking_weights_t` は個別銘柄 × テーマのロングショート・ウェイトである。
- `theme_exposure_t` の対角成分が1、非対角成分が0近傍なら、純化テーマファクター複製ポートフォリオとしての制約を満たしている。
- `barra_exposure_t` が0近傍なら、Barra既存因子への不要なベットを抑制できている。
- 履歴評価を有効化した場合、`replicated_returns_residual` と `target_theme_returns_g` の correlation、beta、tracking error を確認する。
- 制約誤差が大きい場合は、銘柄数不足、制約数過多、`Sigma_t` の特異性、`Z_t` のrank不足を診断テーブルで確認する。